# ЛР №2 — Теория к лабораторной работе
## «Реализация многопоточности и параллельного выполнения в распределённом приложении»

**Предметная область (вариант 14):** «Заказ и доставка цветов» — продолжение ЛР №1.
**Общий ресурс:** `Bouquet.AvailableCount` (количество доступных экземпляров букета).
**Особенность ЛР №2:** проблема конкурентного доступа (**Race Condition**) *намеренно не исправляется* — она создаётся и демонстрируется. Механизмы синхронизации — тема ЛР №3.

Все утверждения в этом конспекте подкреплены фрагментами реального кода решения
`Лабы\Лаб2` (проекты `Server`, `Client`, `Shared`, `Server.Tests`).

> **Как пользоваться конспектом.** Ячейки с кодом — это выдержки из файлов решения
> с путями и пояснениями. Запускать ноутбук не требуется: он используется как
> справочный материал для подготовки к сдаче.

### Содержание

| № | Раздел | Что доказывает |
|---|---|---|
| 1 | Процесс, поток, Task | различие уровней абстракции многозадачности в .NET |
| 2 | Асинхронный приём подключений | почему сервер не блокируется на клиенте |
| 3 | Обработка одного клиента | изоляция ошибок, несколько запросов на соединении |
| 4 | Асинхронный ввод-вывод и фрейминг | почему `await` освобождает поток |
| 5 | Журнал START/END | инструмент доказательства параллельности |
| 6 | Экспериментальная задержка `delayMs` | управляемое время обработки для замера |
| 7 | Race Condition | общий ресурс, окно гонки, потерянное обновление |
| 8 | Ложные «защиты» | почему `ConcurrentQueue` и `volatile` не спасают |
| 9 | Тесты как формальное доказательство | проверяемые критерии параллельности и гонки |
| 10 | Порядок ручного эксперимента | воспроизводимая процедура |
| 11 | Соответствие требованиям и защита | карта «требование → код → результат» |

---
## 1. Процесс, поток, Task

### 1.1 Процесс

**Процесс** — запущенная копия программы (`dotnet run --project Server`,
`dotnet run --project Client`). Каждый процесс изолирован: своё адресное
пространство памяти, свои переменные, своя куча. Поэтому `AvailableCount`
на сервере и на клиенте — это **разные** переменные в **разных** процессах;
клиент физически не может изменить значение, которое читает сервер.

Обмен идёт только по сети: клиент сериализует запрос в JSON, отправляет по TCP,
сервер десериализует, меняет свой объект и возвращает JSON-ответ.

In [ ]:
// Server/Program.cs — точка входа серверного процесса
var port = args.Length > 0 && int.TryParse(args[0], out int p) ? p : Server.DefaultPort;
var server = new Server(port);

Console.CancelKeyPress += (_, e) =>
{
    e.Cancel = true;
    Console.WriteLine("[Сервер] Остановка...");
    server.Stop();
    Environment.Exit(0);
};

// ЛР №2: сервер работает асинхронно (async/await), приём подключений не блокирует
// обработку — каждый клиент обслуживается в отдельной Task.
await server.StartAsync();

Обратите внимание на `await server.StartAsync();` в top-level-коде: главный поток
процесса не «крутит» цикл вручную — он передаёт управление асинхронному циклу
сервера и остаётся свободным (в `Main` при этом нет блокирующего `Wait`).

### 1.2 Поток (Thread)

**Поток** — единица исполнения внутри процесса. Потоки одного процесса разделяют
кучу, но выполняют инструкции последовательно. В сервере .NET несколько потоков
существуют всегда: пул потоков (`ThreadPool`) обслуживает продолжения (continuation)
после `await` и задачи из `Task.Run`.

**Номер потока виден в журнале сервера** — это и есть первичный признак того,
что работа действительно распределяется между потоками. В коде он берётся из
`Environment.CurrentManagedThreadId`.

### 1.3 Task

**Task** — единица *асинхронной* работы в .NET. Ключевое отличие от потока:
Task не имеет своего выделенного потока. Пока Task ожидает ввода-вывода
(`await Task.Delay`, `await stream.ReadAsync`), поток освобождается и используется
для других Task. Это и называют **асинхронной параллельностью (конкурентностью)**
вместо «настоящей» параллельности на ядрах.

| Понятие | Что это | Где в коде |
|---|---|---|
| Process | отдельная программа, своя память | `dotnet run --project Server` / `Client` |
| Thread | последовательность исполнения инструкций | `Environment.CurrentManagedThreadId` в логе |
| Task | отложенная/параллельная операция | `Task.Run(...)`, `HandleClientAsync`, `Task.Delay` |

In [ ]:
// Server/Server.cs — запись журнала: кто, когда, в какой Task, в каком потоке
var entry = new ProcessingLogEntry(
    DateTime.Now, clientId, phase, operation,
    Task.CurrentId, Environment.CurrentManagedThreadId, elapsedMs);

Journal.Enqueue(entry);

string elapsed = elapsedMs is double ms ? $" ({ms / 1000:0.000} с)" : "";
Console.WriteLine(
    $"[{entry.Timestamp:HH:mm:ss.fff}] Client {clientId} → {phase,-10} {operation,-22} " +
    $"Task={entry.TaskId?.ToString() ?? "-"} Thread={entry.ThreadId}{elapsed}");

**Читаем лог так:**

* `Task=4 Thread=6` — обработка шла в Task №4, и на момент логирования
  исполнялась потоком №6. **Task ≠ Thread**: номер Task идентифицирует *операцию*,
  номер потока — *исполнителя*. Один Task может по ходу работы переключаться
  между потоками пула (в том числе при возврате управления после `await`),
  и наоборот — один поток может обслуживать несколько Task.
* `Task=-` — запись сделана вне Task (например, `CONNECT`/`DISCONNECT`,
  где `Task.CurrentId` возвращает `null`).
* Время в формате `HH:mm:ss.fff` (миллисекунды) принципиально: без миллисекунд
  перекрытие интервалов длительностью в секунды невозможно зафиксировать.

---
## 2. Асинхронный приём подключений — ядро ЛР №2

Главный вопрос лабораторной: *как сервер принимает второго клиента, пока первый
ещё обрабатывается?* Ответ — в асинхронном приёме и выносе обработки клиента
в отдельную Task.

In [ ]:
// Server/Server.cs
public async Task StartAsync(CancellationToken ct = default)
{
    _listener.Start();
    _running = true;
    Console.WriteLine($"[Сервер] Запущен и ожидает подключений на порту {Port}...");

    try
    {
        while (_running)
        {
            // Асинхронный приём: сервер не блокируется ни на одном клиенте
            TcpClient client = await _listener.AcceptTcpClientAsync(ct);

            int clientId = Interlocked.Increment(ref _clientCounter);
            Console.WriteLine($"[Сервер] Новое подключение: {client.Client.RemoteEndPoint} → Client {clientId}");

            // Обработка клиента — в отдельной Task (Требование 2)
            _ = Task.Run(() => HandleClientAsync(client, clientId), CancellationToken.None);
        }
    }
    catch (Exception ex) when (ex is SocketException or ObjectDisposedException or InvalidOperationException)
    {
        // Нормальная остановка сервера (Stop() закрывает listener)
    }
    catch (OperationCanceledException)
    {
        // Отмена по токену
    }
}

### Разбор по шагам

1. **`await _listener.AcceptTcpClientAsync(ct)`** — вместо блокирующего `AcceptTcpClient()`.
   Пока новых подключений нет, метод *не удерживает* поток: он возвращает
   незавершённую Task, а поток освобождается для других операций. Как только
   пришёл клиент — управление возвращается в цикл.
   Это даёт главный эффект: **клиент №2 может подключиться, пока клиент №1
   ещё в работе**.

2. **`int clientId = Interlocked.Increment(ref _clientCounter);`** — выдача
   уникального номера клиента. Простое `_clientCounter++` здесь небезопасно:
   операция «прочитать, увеличить, записать» состоит из трёх шагов, и два
   вызова из разных потоков (пула) могут перемешаться и выдать один и тот же номер.
   `Interlocked.Increment` выполняет увеличение атомарно на уровне процессора
   (атомарная RMW-инструкция) — без блокировки и без `lock`.

3. **`_ = Task.Run(() => HandleClientAsync(client, clientId), ...)`** — обработка
   клиента **отсоединяется** от цикла приёма.
   * `Task.Run` ставит делегат в пул потоков и возвращает Task немедленно →
     цикл сразу возвращается к `AcceptTcpClientAsync`.
   * Символ `_ =` — сознательное «fire and forget»: Task намеренно не ожидается,
     потому что ожидать её нельзя — иначе цикл приёма снова стал бы
     последовательным.
   * `CancellationToken.None` — обработка уже принятого клиента **не отменяется**
     при остановке сервера: гарантируется «корректное завершение» (graceful).

4. **`_running` объявлен как `volatile bool`** — флаг читается в одном потоке
   (цикл приёма) и пишется в другом (главный поток из `Console.CancelKeyPress`
   → `server.Stop()`). `volatile` запрещает компилятору и CPU переставлять
   и кэшировать чтение, гарантируя видимость изменения между потоками.

In [ ]:
// Server/Server.cs — поля и конструктор
private readonly TcpListener _listener;
private readonly RequestHandler _handler;
private volatile bool _running;
private int _clientCounter;

/// <summary>Бизнес-логика с общим ресурсом (для тестов и экспериментов ЛР №2).</summary>
public BusinessLogic Logic { get; }

/// <summary>Журнал обработки: START/END по каждому клиенту с временными метками.</summary>
public ConcurrentQueue<ProcessingLogEntry> Journal { get; } = new();

/// <summary>Фактический порт (полезен при порте 0 — эфемерный порт в тестах).</summary>
public int Port => ((IPEndPoint)_listener.LocalEndpoint).Port;

### 2.1 Сравнение: как было бы в синхронном варианте

```csharp
// ТАК БЫЛО (ЛР №1) — последовательная обработка, второй клиент ждёт
while (true)
{
    TcpClient client = _listener.AcceptTcpClient();   // блокирует поток
    HandleClient(client);                            // блокирует до конца
}                                                   // ← второй клиент ждёт здесь

// ТАК СТАЛО (ЛР №2) — параллельная обработка
while (_running)
{
    TcpClient client = await _listener.AcceptTcpClientAsync(ct);  // не блокирует
    _ = Task.Run(() => HandleClientAsync(client, clientId));      // уходит в фон
}                                                   // ← цикл сразу свободен
```

Ожидаемое время при двух клиентах с задержкой 10 с:
* последовательно — ≈ 20 с (10 + 10);
* параллельно — ≈ 10–11 с (второй стартует почти сразу после первого).

Это и есть измеримый критерий параллельности, который потом проверяется тестом.

---
## 3. Обработка одного клиента

Каждое подключение обслуживается собственной Task — `HandleClientAsync`.

In [ ]:
// Server/Server.cs
private async Task HandleClientAsync(TcpClient client, int clientId)
{
    using TcpClient c = client;
    Log(clientId, "CONNECT", "-");

    try
    {
        NetworkStream stream = c.GetStream();
        while (_running)
        {
            string json = await MessageFramer.ReadMessageAsync(stream);
            string operation = RequestHandler.TryGetOperationName(json);

            // Требование 3: фиксируем начало обработки
            var stopwatch = Stopwatch.StartNew();
            Log(clientId, "START", operation);

            Response response = await _handler.HandleAsync(json);

            stopwatch.Stop();
            Log(clientId, "END", operation, stopwatch.Elapsed.TotalMilliseconds);

            string responseJson = System.Text.Json.JsonSerializer.Serialize(response, Json.Options);
            await MessageFramer.WriteMessageAsync(stream, responseJson);
        }
    }
    catch (EndOfStreamException)
    {
        // Клиент закрыл соединение — это нормально (Требование 9)
        Log(clientId, "DISCONNECT", "-");
    }
    catch (IOException ex)
    {
        Console.WriteLine($"[Сервер] Client {clientId}: ошибка сетевого взаимодействия: {ex.Message}");
    }
    catch (Exception ex)
    {
        // Ошибка одного клиента не приводит к аварийному завершению сервера (Требование 9)
        Console.WriteLine($"[Сервер] Client {clientId}: ошибка обработки: {ex.Message}");
    }
}

### 3.1 Что здесь важно для защиты

* **`using TcpClient c = client;`** — ресурс соединения освобождается при выходе
  из метода, в том числе при исключении. Без `using` сокеты накапливались бы,
  и при большом числе клиентов сервер упёрся бы в лимит дескрипторов.
* **Цикл `while (_running)` внутри клиента** — одно TCP-соединение может
  обслуживать несколько запросов последовательно, пока клиент не закроет поток.
  Параллелизм обеспечивается не этим циклом, а тем, что сам `HandleClientAsync`
  уже выполняется в отдельной Task.
* **Три уровня `catch`** — от «ожидаемых» событий к «неожидаемым»:
  1. `EndOfStreamException` — противоположная сторона закрыла соединение
     (`MessageFramer` бросает его при `read <= 0`). Это нормальный сценарий:
     клиент завершил работу. Сервер логирует `DISCONNECT` и корректно выходит
     **только из этого клиента**.
  2. `IOException` — сетевой сбой конкретного соединения.
  3. `Exception` — любая бизнес-ошибка.

  Общий принцип: **исключение гасится на границе Task клиента**, поэтому
  «упавший» клиент не валит ни цикл приёма, ни сервер. Без этого `catch`
  необработанное исключение из фоновой Task привело бы к
  `TaskScheduler.UnobservedTaskException` (в .NET Framework — к падению процесса).
* **`Stopwatch`** — высокоточный секундомер (`System.Diagnostics.Stopwatch`).
  Даёт `Elapsed.TotalMilliseconds` в лог `END`.

In [ ]:
// Server/Server.cs — остановка сервера
public void Stop()
{
    _running = false;   // выход из while в StartAsync и из while в HandleClientAsync
    _listener.Stop();   // закрытие listener → AcceptTcpClientAsync бросает исключение,
                        // которое перехватывается в StartAsync
}

---
## 4. Асинхронный ввод-вывод и фрейминг сообщений

ЛР №1 использовала синхронные `ReadMessage`/`WriteMessage`. В ЛР №2 они
заменены на асинхронные версии — именно они обеспечивают «неблокирующее
ожидание» на уровне ввода-вывода.

In [ ]:
// Shared/Protocol/MessageFramer.cs
public static async Task WriteMessageAsync(Stream stream, string message, CancellationToken ct = default)
{
    var body = Encoding.UTF8.GetBytes(message);
    var header = BitConverter.GetBytes(body.Length);
    await stream.WriteAsync(header, ct);
    await stream.WriteAsync(body, ct);
    await stream.FlushAsync(ct);
}

public static async Task<string> ReadMessageAsync(Stream stream, CancellationToken ct = default)
{
    byte[] header = new byte[4];
    await ReadExactlyAsync(stream, header, ct);
    return await ReadBodyAsync(stream, header, ct);
}

private static async Task ReadExactlyAsync(Stream stream, Memory<byte> buffer, CancellationToken ct)
{
    int offset = 0;
    while (offset < buffer.Length)
    {
        int read = await stream.ReadAsync(buffer[offset..], ct);
        if (read <= 0)
        {
            throw new EndOfStreamException("Соединение закрыто другой стороной");
        }

        offset += read;
    }
}

### 4.1 Формат кадра

```
[ 4 байта длины (little-endian, int32) ][ UTF-8 JSON ]
```

TCP — потоковый протокол без границ сообщений: один `Read` может вернуть
полсообщения или несколько сообщений, склеенных вместе. Префикс длины
решает задачу **кадрирования (framing)**.

### 4.2 Почему `ReadExactlyAsync` — цикл, а не один `ReadAsync`

`Stream.ReadAsync` не гарантирует заполнение буфера: TCP-сегменты могут
прийти по отдельности. Пока `offset < buffer.Length`, читаем в оставшуюся
часть буфера `buffer[offset..]` (диапазонная проекция `Memory<byte>`).
При `read <= 0` — соединение закрыто, бросается `EndOfStreamException`
(именно его ловит `HandleClientAsync` как нормальный `DISCONNECT`).

### 4.3 Защита от некорректного кадра

`length <= 0 || length > MaxMessageSize` (`8 МБ`) → `IOException`.
Это защита от «мусорных» 4 байт в начале потока: иначе сервер попытался бы
выделить массив неадекватного размера.

### 4.4 Смысл `await` с точки зрения многозадачности

`await stream.ReadAsync(...)` в ожидании данных **не занимает поток**:
у Task есть продолжение (continuation), которое будет выполнено, когда данные
придут. Поток на это время может обслуживать другого клиента.
Отсюда — возможность обслуживать тысячи клиентов небольшим числом потоков.

---
## 5. Журнал START/END как доказательство параллельности

Параллельность — понятие, которое требует наблюдаемого признака.
В этой ЛР таким признаком служит `Journal`: журнал обработки, куда пишется
**каждый** вход и выход из обработки с временной меткой.

In [ ]:
// Server/Server.cs
public sealed record ProcessingLogEntry(
    DateTime Timestamp,
    int ClientId,
    string Phase,      // CONNECT / START / END / DISCONNECT
    string Operation,  // имя операции из JSON
    int? TaskId,
    int ThreadId,
    double? ElapsedMs);

Используется **`record`** — неизменяемый ссылочный тип с автоматическим
`ToString`, сравнением по значению и `Deconstruct`. Идеально для записи журнала:
запись нельзя изменить «задним числом» после добавления в очередь, а тесты
могут сравнивать её по значениям полей.

In [ ]:
// Server/Server.cs — потокобезопасная очередь журнала
private readonly ConcurrentQueue<ProcessingLogEntry> Journal { get; } = new();

// Запись из ЛЮБОГО числа Task одновременно
Journal.Enqueue(entry);

// Чтение снимка для теста (ToArray — снимок на момент вызова)
var journal = host.Server.Journal.ToArray();

**Почему `ConcurrentQueue`, а не `List`/`Queue`?** `List<T>` и `Queue<T>`
не потокобезопасны: одновременные `Enqueue` из разных Task приводят к
повреждению внутреннего массива (потеря записей, битые индексы, гонка
за `_size`). `ConcurrentQueue<T>` реализован через связные узлы
и lock-free алгоритмы — безопасен для множества писателей и читателей.
В Journal пишут все Task клиентов одновременно — значит потокобезопасность
коллекции здесь обязательна (это «правильная» защита, в отличие от п. 8).

### 5.1 Что именно доказывает журнал

Для каждого клиента получаем интервал обработки
`[START.Timestamp, END.Timestamp]`. Параллельность доказана, если интервалы
**перекрываются**: `max(START1, START2) < min(END1, END2)`.

Пример вывода (задержка 10 с, второй клиент подключается сразу):

```
[10:15:01.120] Client 1 → START      CreateOrder   Task=4 Thread=6
[10:15:01.625] Client 2 → START      CreateOrder   Task=7 Thread=8
[10:15:11.125] Client 1 → END        CreateOrder   Task=4 Thread=6 (10.005 с)
[10:15:11.631] Client 2 → END        CreateOrder   Task=7 Thread=8 (10.006 с)
```

Видно четыре факта одновременно:
1. `START(Client 2)` раньше `END(Client 1)` → интервалы перекрыты → **параллельно**;
2. у клиентов **разные Task** (4 и 7) → это независимые единицы работы;
3. `Thread` у каждого свой в момент логирования → работа распределяется по потокам;
4. общее время ≈ 10 с вместо ≈ 20 с → выигрыш фактический, а не «надпись в логе».

Разные номера `Task` при этом **не означают**, что Task обязаны работать
на разных ядрах: параллелизм здесь — на уровне логики (одновременное
*существование* обработок), а не на уровне CPU.

---
## 6. Экспериментальная задержка `delayMs`

Чтобы «увидеть» параллельность, нужно сделать обработку достаточно длинной.
Задержка вводится полем `delayMs` в `payload` **любого** запроса — так
эксперимент не зависит от бизнес-операции.

In [ ]:
// Server/RequestHandler.cs
int delayMs = ReadDelayMs(request.Payload);
if (delayMs > 0)
{
    // Искусственное увеличение времени обработки (Требование 4).
    // Задержка нужна только для проведения эксперимента.
    Console.WriteLine($"[Сервер] Экспериментальная задержка обработки: {delayMs} мс...");
    await Task.Delay(delayMs, ct);
}

return await _logic.ExecuteAsync(request, ct);

In [ ]:
// Server/RequestHandler.cs — валидация задержки
public const int MaxDelayMs = 300_000;

private static int ReadDelayMs(JsonElement? payload)
{
    if (payload is not JsonElement p ||
        p.ValueKind != JsonValueKind.Object ||
        !p.TryGetProperty("delayMs", out var v))
    {
        return 0;
    }

    int delayMs;
    if (v.ValueKind == JsonValueKind.Number && v.TryGetInt32(out int n)) delayMs = n;
    else if (v.ValueKind == JsonValueKind.String && int.TryParse(v.GetString(), out int s)) delayMs = s;
    else throw new OperationException(ErrorCodes.ValidationError, "Поле 'delayMs' должно быть целым числом (миллисекунды)");

    if (delayMs < 0 || delayMs > MaxDelayMs)
    {
        throw new OperationException(ErrorCodes.ValidationError,
            $"Поле 'delayMs' должно быть в диапазоне 0..{MaxDelayMs} мс");
    }
    return delayMs;
}

**Почему `await Task.Delay`, а не `Thread.Sleep`?** `Thread.Sleep` **блокирует**
поток целиком: поток ОС не делает ничего полезного все 10 секунд.
`await Task.Delay` только *приостанавливает* Task — поток возвращается в пул и
обслуживает других клиентов. Именно поэтому в эксперименте два 10-секундных
запроса дают ~10 с суммарно, а не ~20 с.

`CancellationToken` передаётся в `Task.Delay` и в `ExecuteAsync`: при остановке
сервера задержки обрываются, а не «досчитываются».
Валидация диапазона `0..300000` защищает и от мусора в payload, и от запроса,
который «подвесит» эксперимент на часы.

In [ ]:
// Server/RequestHandler.cs — асинхронный диспетчер с изоляцией ошибок
public async Task<Response> HandleAsync(string requestJson, CancellationToken ct = default)
{
    try
    {
        Request? request;
        try
        {
            request = JsonSerializer.Deserialize<Request>(requestJson, Json.Options);
        }
        catch (JsonException)
        {
            return Response.Fail(ErrorCodes.ValidationError, "Некорректный формат JSON-запроса");
        }

        if (request is null || string.IsNullOrWhiteSpace(request.Operation))
        {
            return Response.Fail(ErrorCodes.ValidationError, "Не указана операция (поле 'operation')");
        }
        // ... чтение delayMs и await Task.Delay(delayMs, ct) ...
        return await _logic.ExecuteAsync(request, ct);
    }
    catch (OperationException ex)        { return Response.Fail(ex.ErrorCode, ex.Message); }
    catch (OperationCanceledException)  { return Response.Fail(ErrorCodes.InternalError, "Обработка запроса была отменена"); }
    catch (Exception ex)                { return Response.Fail(ErrorCodes.InternalError, $"Внутренняя ошибка сервера: {ex.Message}"); }
}

/// <summary>Синхронная обработка (обёртка над HandleAsync).</summary>
public Response Handle(string requestJson) => HandleAsync(requestJson).GetAwaiter().GetResult();

Обратите внимание на разновидность `catch`: `OperationException` — это
**ожидаемая** бизнес-ошибка с кодом протокола (`ValidationError`, `NotFound`,
`OutOfStock`, `UnknownOperation`), она возвращается клиенту как структурированный
`Response.Fail`. Неожиданные ошибки превращаются в `InternalError`.
Ни одна из них не выходит наружу Task — сервер продолжает работу.

`GetAwaiter().GetResult()` в синхронных обёртках `Handle` / `Execute` —
это «мост» для unit-тестов и синхронных вызовов. Он не приводит к deadlock
в типичном случае, потому что здесь нет внешнего контекста синхронизации
(`SynchronizationContext`), в отличие от UI-приложений (WinForms/WPF).

---
## 7. Race Condition — конкурентный доступ к общему ресурсу

### 7.1 Определение

**Race condition (состояние гонки)** — ситуация, когда результат работы
программы зависит от **непредсказуемого порядка** выполнения конкурирующих
операций. Поведение системы становится недетерминированным: при одинаковых
входных данных возможны разные результаты.

Классические условия гонки (должны выполняться одновременно):
1. **Общий изменяемый ресурс** (shared mutable state).
2. **Конкурентный доступ** к нему из нескольких Task/потоков.
3. **Неатомарная последовательность** «проверить → решить → изменить»,
   внутри которой ресурс может измениться.

Все три условия в этой ЛР выполнены намеренно.

### 7.2 Общий ресурс

Ресурс — количество доступных экземпляров букета: `Bouquet.AvailableCount`.
Он **один на весь серверный процесс** и разделяется всеми клиентами
(в отличие от данных клиента, каждый из которых работает в своём процессе).

In [ ]:
// Shared/Models/Bouquet.cs (фрагмент) — общий ресурс предметной области
public class Bouquet
{
    public int Id { get; set; }
    public string Name { get; set; } = "";
    public int AvailableCount { get; set; }   // <-- ОБЩИЙ РЕСУРС ЛР №2
    public List<FlowerItem> Flowers { get; set; } = new();
}

In [ ]:
// Server/InMemoryDataStore.cs — демо-букет для эксперимента
// Букет для эксперимента ЛР №2: количество общего ресурса = 1
Bouquets.Add(new Bouquet
{
    Id = NextBouquetId(),
    Name = "Новогодний люкс",
    AvailableCount = 1,
    Flowers = { new FlowerItem { FlowerName = "Роза", Count = 15, PricePerUnit = 200m } }
});

### 7.3 Три шага операции `ReserveBouquet` — источник гонки

In [ ]:
// Server/BusinessLogic.cs
// --- Шаг 1: ПРОВЕРКА доступности (чтение общего ресурса, без блокировки) ---
int available = bouquet.AvailableCount;
Console.WriteLine(
    $"[Сервер] [Race Condition] READ: Букет #{bouquetId} «{bouquet.Name}» — " +
    $"доступно {available} шт. (клиент читает общий ресурс)");

// --- Шаг 2: ОКНО ГОНКИ — искусственная задержка между проверкой и изменением ---
if (raceDelayMs > 0)
{
    await Task.Delay(raceDelayMs, ct);
}

// --- Шаг 3: РЕШЕНИЕ принимается по УСТАРЕВШЕМУ значению (проблема не исправлена!) ---
if (available <= 0)
{
    throw new OperationException(ErrorCodes.OutOfStock,
        $"Букет «{bouquet.Name}» недоступен для бронирования: свободных экземпляров не осталось");
}

bouquet.AvailableCount--; // запись без синхронизации — возможна потеря обновления
Console.WriteLine(
    $"[Сервер] [Race Condition] WRITE: Букет #{bouquetId} — установлено " +
    $"{bouquet.AvailableCount} шт. (клиент изменил общий ресурс)");

Обратите внимание на **содержательную ошибку** шага 3: проверка
`if (available <= 0)` использует **локальную переменную `available`**, прочитанную
в шаге 1, а не текущее значение `bouquet.AvailableCount`. Между шагом 1 и шагом 3
другой клиент уже изменил ресурс, но решение принимается по устаревшему снимку.
Это и есть **lost update (потерянное обновление)**.

### 7.4 Хронология эксперимента (режим гонки)

```
время ↓        Client 1 (Task A)                          Client 2 (Task B)
              READ  available = 1
              ── окно гонки: await Task.Delay(2000) ──►    READ  available = 1   ← тоже 1!
              WRITE AvailableCount--  → 0                  ── окно гонки ──►
                                                          WRITE AvailableCount-- → -1
```

Ключевой факт: оба клиента прочитали `1` и оба признали ресурс доступным.
Ресурс был **один**, а успешных бронирований — **два**.

In [ ]:
// Воспроизведение логики шагов 1-3 в псевдокоде
int available = bouquet.AvailableCount;   // Шаг 1: READ  (снимок значения)
await Task.Delay(raceDelayMs, ct);       // Шаг 2: окно гонки (конкурентный вход)
if (available <= 0) throw;               // Шаг 3: проверка по УСТАРЕВШЕМУ снимку
bouquet.AvailableCount--;                // Шаг 3: WRITE по УСТАРЕВШЕМУ снимку

### 7.5 Модель гонки для проверки на бумаге (псевдокод)

```
время ↓        Client 1 (Task A)                          Client 2 (Task B)
              READ  available = 1
              ── окно гонки: await Task.Delay(2000) ──►    READ  available = 1   ← тоже 1!
              WRITE AvailableCount--  → 0                  ── окно гонки ──►
                                                          WRITE AvailableCount-- → -1

Результат: РЕСУРС = 1, УСПЕШНЫХ БРОНИРОВАНИЙ = 2, AvailableCount = -1
```

Порядок возможен **только** при параллельном исполнении: при последовательной
обработке второй `READ` выполнился бы уже после первого `WRITE` и увидел бы 0 —
тогда второй клиент получил бы `OutOfStock`, а не `SUCCESS`.

### 7.6 Реальный вывод сервера (эксперимент на двух клиентах)

```
[Сервер] [Race Condition] READ: Букет #3 «Новогодний люкс» — доступно 1 шт. (клиент читает общий ресурс)
[Сервер] [Race Condition] READ: Букет #3 «Новогодний люкс» — доступно 1 шт. (клиент читает общий ресурс)
[Сервер] [Race Condition] WRITE: Букет #3 — установлено 0 шт. (клиент изменил общий ресурс)
[Сервер] [Race Condition] WRITE: Букет #3 — установлено -1 шт. (клиент изменил общий ресурс)
```

**Оба клиента получили `SUCCESS`**, хотя экземпляр был только один, а
`AvailableCount` ушёл в минус. Это и есть наблюдаемое доказательство гонки.

### 7.7 Операции, обслуживающие эксперимент

In [ ]:
// Server/BusinessLogic.cs
public Response SetBouquetAvailability(JsonElement? payload)   // установить ресурс перед гонкой
{
    // ... валидация bouquetId и count (count >= 0) ...
    bouquet.AvailableCount = count;
    return Response.Ok(new { bouquetId, bouquet.Name, AvailableCount = bouquet.AvailableCount }, ...);
}

public Response GetBouquetStatus(JsonElement? payload)           // посмотреть состояние ресурса
{
    if (/* передан bouquetId */)
    {
        return Response.Ok(new { bouquetId, bouquet.Name, AvailableCount = bouquet.AvailableCount }, ...);
    }
    var all = _db.Bouquets.Select(b => new { b.Id, b.Name, AvailableCount = b.AvailableCount }).ToList();
    return Response.Ok(new { bouquets = all }, "Состояние общего ресурса получено");
}

Роли операций:

* `SetBouquetAvailability` — подготовка: выставляет `AvailableCount = 1`,
  чтобы ресурс был заведомо дефицитным (иначе гонку не пронаблюдать).
* `ReserveBouquet` — сама гонка; параметр `raceDelayMs` (в тестах 2000 мс)
  задаёт ширину окна.
* `GetBouquetStatus` — наблюдение: только эта операция показывает итоговое
  значение ресурса, по которому и обнаруживается `AvailableCount = -1`.

`GetBouquetStatus` выполняется **после** завершения обоих бронирований,
поэтому «видит» уже итоговое значение, а не промежуточное.

---
## 8. Ложные «защиты»: почему код НЕ синхронизирован

На защите часто звучит вопрос: «Ведь журнал уже `ConcurrentQueue` — почему
нельзя считать, что доступ синхронизирован?» Разберём, что защищает,
а что нет.

| Элемент кода | Потокобезопасен? | Защищает ли `AvailableCount`? |
|---|---|---|
| `Journal` = `ConcurrentQueue<ProcessingLogEntry>` | **да** | **нет** — это отдельный объект |
| `Interlocked.Increment(ref _clientCounter)` | **да** (атомарно) | **нет** — только счётчик клиентов |
| `volatile bool _running` | **да** (видимость) | **нет** — только флаг остановки |
| `int AvailableCount { get; set; }` | **нет** | **нет** — здесь гонка и остаётся |

### 8.1 Потокобезопасность ≠ атомарность бизнес-операции

`ConcurrentQueue` гарантирует, что **каждая запись** журнала попадёт в очередь
целиком и без повреждений. Но он ничего не говорит о *согласованности
нескольких* операций, выполненных подряд (прочитал → сравнил → записал).
Потокобезопасная коллекция — это про «каждая операция не повредит структуру»,
а не про «последовательность операций атомарна».

### 8.2 `volatile` — это про видимость, а не про атомарность

`volatile bool _running` гарантирует, что запись одного потока станет видна
другому. Но `AvailableCount` объявлен как обычный `int` без `volatile`.
Даже если бы он был `volatile`, это не помогло бы: проблема не в «невидимости»
значения, а в том, что **между чтением и записью** другой Task успел изменить
ресурс. Нужна атомарность *последовательности* — то есть блокировка
(в ЛР №3) или атомарная операция сравнения-и-замены.

### 8.3 Что в ЛР №3 (не здесь!)

Правильное исправление — сделать «проверить и уменьшить» одной неделимой
операцией. Варианты, которые разбираются в следующей лабораторной:
`lock`/`Monitor` вокруг блока READ–WRITE, `Interlocked.CompareExchange`
(атомарное условное уменьшение), `SemaphoreSlim`, `Immutable*` и
потокобезопасные коллекции для состояния.

**В ЛР №2 этого делать нельзя** — по условию задания гонка демонстрируется,
а не устраняется. Исправленный код прошёл бы тест
`TwoClients_ReserveLastBouquet_BothSucceed_RaceConditionOccurs`
(там ожидается `availableCount == -1`), и это считалось бы ошибкой.

---
## 9. Тесты как формальное доказательство

ЛР №1 тестировала парсинг/валидацию. В ЛР №2 добавлены тесты, которые
доказывают **параллельность** и **наличие гонки**. Все они поднимают
настоящий сервер на эфемерном порту.

In [ ]:
// Server.Tests/ParallelProcessingTests.cs
public sealed class TestServerHost : IDisposable
{
    public ShopServer Server { get; }
    private readonly Task _runLoop;

    public TestServerHost()
    {
        Server = new ShopServer(0); // 0 — эфемерный порт, выберется автоматически
        // До первого await (AcceptTcpClientAsync) тело выполнится синхронно,
        // поэтому к возврату listener уже запущен и Port доступен.
        _runLoop = Server.StartAsync();
    }

    public NetworkClient CreateClient() => new("127.0.0.1", Server.Port);

    public void Dispose()
    {
        Server.Stop();
        try { _runLoop.Wait(TimeSpan.FromSeconds(3)); }
        catch (AggregateException) { /* нормальная остановка */ }
    }
}

Два приёма, которые стоит уметь объяснить:

1. **Порт `0`** — «эфемерный» порт: ОС выбирает свободный сам, и `Server.Port`
   возвращает фактический. Так тесты не конфликтуют с уже запущенным вручную
   сервером на порту по умолчанию.
2. **Зачем `_runLoop`?** `StartAsync()` — бесконечный цикл, его нельзя
   `await`-ить в тесте. Task сохраняется, а в `Dispose` ожидается с таймаутом
   3 с, чтобы тестовый процесс корректно освобождал порт.
   `catch (AggregateException)` — `Wait` «заворачивает» исключение остановки
   сервера в `AggregateException`; это штатная ситуация, а не ошибка.

In [ ]:
// Тест 1. Один клиент — базовая корректность
[Fact]
public async Task OneClient_RequestIsProcessedCorrectly()
{
    using var host = new TestServerHost();
    using var client = host.CreateClient();

    Response response = await client.SendAsync(
        RequestBuilder.CreateOrder(1, 1, "ул. Ленина, 10", "2026-10-05"));

    Assert.True(response.Success, response.Message);
    Assert.NotNull(response.Data);

    var journal = host.Server.Journal.ToArray();
    Assert.Contains(journal, e => e.ClientId == 1 && e.Phase == "START" && e.Operation == "CreateOrder");
    Assert.Contains(journal, e => e.ClientId == 1 && e.Phase == "END" && e.Operation == "CreateOrder");

    var start = journal.Single(e => e.ClientId == 1 && e.Phase == "START");
    var end = journal.Single(e => e.ClientId == 1 && e.Phase == "END");
    Assert.True(start.Timestamp <= end.Timestamp, "START должен быть не позже END");
}

Проверяется базовый инвариант журнала: `START ≤ END` для одного клиента.
`Single` требует, чтобы запись была **ровно одна** — это дополнительно
доказывает, что журнал не задублирован.

In [ ]:
// Тест 2. Два клиента одновременно — Task.WhenAll
[Fact]
public async Task TwoClients_BothRequestsAreProcessed()
{
    using var host = new TestServerHost();
    using var client1 = host.CreateClient();
    using var client2 = host.CreateClient();

    // Оба запроса отправляются одновременно (Task.WhenAll)
    Task<Response> task1 = client1.SendAsync(
        RequestBuilder.CreateOrder(1, 1, "ул. Ленина, 10", "2026-10-05"));
    Task<Response> task2 = client2.SendAsync(
        RequestBuilder.CreateOrder(2, 2, "пр. Мира, 25", "2026-10-06"));

    Response[] results = await Task.WhenAll(task1, task2);

    Assert.All(results, r => Assert.True(r.Success, r.Message));

    // Сервер выделил двум клиентам разные идентификаторы и обработал обоих
    var journal = host.Server.Journal.ToArray();
    int[] clientIds = journal.Where(e => e.Phase == "END").Select(e => e.ClientId)
                             .Distinct().OrderBy(x => x).ToArray();
    Assert.Equal(new[] { 1, 2 }, clientIds);
}

`Task.WhenAll(t1, t2)` — «запустить обе Task и дождаться обеих»: возвращает
`Task<Response[]>` с результатами в порядке аргументов. Важно: клиенты
**не ждут друг друга** — `SendAsync` у каждого уходит в сеть сразу.
Параллелизм создаётся на стороне сервера (одна Task на подключение),
а `WhenAll` лишь синхронизирует тест.

In [ ]:
// Тест 3. Доказательство параллельности: задержка 10 с, второй стартует через 0.5 с
Response[] results = await Task.WhenAll(task1, task2);

// Client 2 начал обрабатываться ДО завершения Client 1 — запросы обрабатывались параллельно
Assert.True(c2Start.Timestamp < c1End.Timestamp,
    "Client 2 начал обработку позже завершения Client 1 — запросы обрабатывались последовательно");

// Общее время близко к 10 с, а не к 20 с (последовательно было бы ~20.5 с)
Assert.True(totalSeconds < 19,
    $"Общее время {totalSeconds:F1} с — запросы обрабатывались не параллельно");

// Перекрытие интервалов обработки: [c1Start..c1End] ∩ [c2Start..c2End] != пусто
DateTime overlapStart = c1Start.Timestamp > c2Start.Timestamp ? c1Start.Timestamp : c2Start.Timestamp;
DateTime overlapEnd = c1End.Timestamp < c2End.Timestamp ? c1End.Timestamp : c2End.Timestamp;
Assert.True(overlapStart < overlapEnd, "Интервалы обработки двух клиентов не перекрываются");

Три независимых критерия, каждый из которых самодостаточен:

1. **Порядок событий**: `START(Client 2) < END(Client 1)`. Если бы обработка была
   последовательной, это неравенство не могло бы выполниться.
2. **Пересечение интервалов**: `max(START) < min(END)` — формальная запись
   перекрытия (пересечение двух интервалов непусто).
3. **Время**: `totalSeconds < 19` при 2 × 10 с. Порог 19 с — с запасом
   (последовательно вышло бы ~20.5 с), но ниже суммарного времени, поэтому
   ложное срабатывание из-за «медленной машины» невозможно.

Критерий 3 — самый практичный, потому что не зависит от точности часов,
а критерии 1–2 показывают, *почему* время такое.

In [ ]:
// Тест 4. Race Condition: два бронирования одного ресурса
[Fact]
public async Task TwoClients_ReserveLastBouquet_BothSucceed_RaceConditionOccurs()
{
    using var host = new TestServerHost();
    using var admin = host.CreateClient();
    using var client1 = host.CreateClient();
    using var client2 = host.CreateClient();

    // Перед экспериментом устанавливаем количество ресурса = 1 (Требование 6)
    Response set = await admin.SendAsync(RequestBuilder.SetBouquetAvailability(1, 1));
    Assert.True(set.Success, set.Message);

    // Client 1 отправляет запрос бронирования
    Task<Response> task1 = client1.SendAsync(RequestBuilder.ReserveBouquet(1, raceDelayMs: 2000));

    // Пока Client 1 находится в обработке (окно гонки), такой же запрос Client 2
    await Task.Delay(300);
    Task<Response> task2 = client2.SendAsync(RequestBuilder.ReserveBouquet(1, raceDelayMs: 2000));

    Response[] results = await Task.WhenAll(task1, task2);

    // Оба клиента получили SUCCESS, хотя ресурс был только один — Race Condition!
    Assert.All(results, r => Assert.True(r.Success, ...));

    // Состояние общего ресурса после эксперимента: 1 - 1 - 1 = -1
    Response status = await admin.SendAsync(RequestBuilder.GetBouquetStatus(1));
    int availableCount = status.Data!.Value.GetProperty("availableCount").GetInt32();
    Assert.True(availableCount == -1, ...);

    // Второй бронирующий начал обрабатываться до завершения первого — окно гонки
    var reserveStarts = host.Server.Journal
        .Where(e => e.Phase == "START" && e.Operation == "ReserveBouquet")
        .OrderBy(e => e.Timestamp).ToArray();
    var reserveEnds = host.Server.Journal
        .Where(e => e.Phase == "END" && e.Operation == "ReserveBouquet")
        .OrderBy(e => e.Timestamp).ToArray();

    Assert.Equal(2, reserveStarts.Length);
    Assert.Equal(2, reserveEnds.Length);
    Assert.True(reserveStarts[1].Timestamp < reserveEnds[0].Timestamp,
        "Второй клиент должен начать бронирование до завершения первого (окно гонки)");
}

Обратите внимание на **конструкцию теста как доказательства**:

* `Assert.True(availableCount == -1)` — ассерт фиксирует **дефект** как
  ожидаемое поведение. Это сознательный «снимок» поведения системы: тест
  сломается, если гонку вдруг устранят. Именно поэтому в комментарии написано
  «проблема намеренно не исправлена».
* Фильтрация журнала по `Operation == "ReserveBouquet"` отделяет бронирования
  от служебных запросов `admin` (у него свой `ClientId`, т.к. он подключился
  раньше). Иначе в `reserveStarts` попали бы записи посторонней операции.
* `await Task.Delay(300)` — гарантирует, что Client 2 войдёт в окно гонки
  Client 1 (у которого `raceDelayMs = 2000`).
* Последний `Assert` доказывает, что гонка вообще возможна только при
  параллельном исполнении: при последовательной обработке второй `START`
  не мог бы оказаться раньше первого `END`.

In [ ]:
// Client/NetworkClient.cs — асинхронная отправка с таймаутом подключения
public async Task<Response> SendAsync(Request request, CancellationToken ct = default)
{
    using var tcp = new TcpClient();
    try
    {
        try
        {
            using var timeoutCts = CancellationTokenSource.CreateLinkedTokenSource(ct);
            timeoutCts.CancelAfter(TimeSpan.FromSeconds(5));
            await tcp.ConnectAsync(_host, _port, timeoutCts.Token);
        }
        catch (OperationCanceledException) when (!ct.IsCancellationRequested)
        {
            throw new ServerUnavailableException($"Сервер {_host}:{_port} не отвечает (таймаут подключения)");
        }
        catch (SocketException ex)
        {
            throw new ServerUnavailableException($"Сервер {_host}:{_port} недоступен: {ex.Message}", ex);
        }

        using NetworkStream stream = tcp.GetStream();
        string requestJson = System.Text.Json.JsonSerializer.Serialize(request, Json.Options);
        await MessageFramer.WriteMessageAsync(stream, requestJson, ct);

        string responseJson = await MessageFramer.ReadMessageAsync(stream, ct);
        Response? response = System.Text.Json.JsonSerializer.Deserialize<Response>(responseJson, Json.Options);
        return response ?? Response.Fail(ErrorCodes.NetworkError, "Пустой ответ сервера");
    }
    ...
}

/// <summary>Синхронная отправка (обёртка над SendAsync).</summary>
public Response Send(Request request) => SendAsync(request).GetAwaiter().GetResult();

`CancellationTokenSource.CreateLinkedTokenSource(ct)` — «связанный» источник
отмены: он сработает, если отменится **любой** из двух: внешний `ct` или наш
5-секундный таймаут. Фильтр `when (!ct.IsCancellationRequested)` в `catch`
различает эти случаи: если отменён внешний токен — это не «сервер недоступен»,
а штатная отмена теста.

Таймаут подключения нужен, чтобы при незапущенном сервере тест падал быстро
с понятным `ServerUnavailableException`, а не висел бы.

---
## 10. Порядок проведения экспериментов (процедура для защиты)

### 10.1 Сборка и запуск

In [ ]:
dotnet build FlowerShop.sln

# терминал 1 — сервер
dotnet run --project Server

# терминал 2 — Client 1
dotnet run --project Client

# терминал 3 — Client 2
dotnet run --project Client

### 10.2 Эксперимент 1 — параллельная обработка

1. В обоих клиентах выбрать пункт меню, создающий заказ **с задержкой 10 с**
   (в payload передаётся `delayMs = 10000`).
2. Отправить запрос из Client 1, **не дожидаясь ответа** отправить из Client 2.
3. Наблюдать вывод сервера: `START(Client 2)` должен появиться **раньше**,
   чем `END(Client 1)`.
4. Замерить общее время: ≈ 10–11 с. При последовательной обработке
   получилось бы ≈ 20 с.

| Параметр | Последовательно | Параллельно (эта ЛР) |
|---|---|---|
| Client 1 | 0 → 10 с | 0 → 10 с |
| Client 2 | 10 → 20 с | 0.5 → 10.5 с |
| Общее время | ≈ 20 с | ≈ 10–11 с |
| Интервалы | не пересекаются | **пересекаются** |

### 10.3 Эксперимент 2 — Race Condition

1. Client 1, пункт **7** (`SetBouquetAvailability`): `bouquetId = 3`, `count = 1`.
2. Client 1, пункт **6** (`ReserveBouquet`): `bouquetId = 3`.
3. **Не дожидаясь ответа** (~3 с окна гонки), Client 2 выполняет пункт **6** с тем же `bouquetId`.
4. Оба клиента получают `SUCCESS`.
5. Client 1, пункт **8** (`GetBouquetStatus`): `AvailableCount = -1`.

```text
Ожидаемый вывод сервера:
[Сервер] [Race Condition] READ: Букет #3 «Новогодний люкс» — доступно 1 шт.
[Сервер] [Race Condition] READ: Букет #3 «Новогодний люкс» — доступно 1 шт.   <- оба увидели 1
[Сервер] [Race Condition] WRITE: Букет #3 — установлено 0 шт.
[Сервер] [Race Condition] WRITE: Букет #3 — установлено -1 шт.
```

**Вывод:** ресурс был один, успешных бронирований два, значение ушло в минус.
Проверка доступности и её изменение не были атомарны — решение принималось
по устаревшему значению (потерянное обновление).

### 10.4 Запуск тестов

In [ ]:
dotnet test FlowerShop.sln

Обязательные тесты ЛР №2 (класс `ParallelProcessingTests`):

| № | Тест | Что доказывает |
|---|---|---|
| 1 | `OneClient_RequestIsProcessedCorrectly` | корректность, START/END в журнале, `START <= END` |
| 2 | `TwoClients_BothRequestsAreProcessed` | `Task.WhenAll`, оба SUCCESS, разные `ClientId` |
| 3 | `TwoClients_WithTenSecondDelay_AreProcessedInParallel` | `START2 < END1`, перекрытие, общее время < 19 с |
| 4 | `TwoClients_ReserveLastBouquet_BothSucceed_RaceConditionOccurs` | оба SUCCESS, `AvailableCount = -1` |

Плюс тесты ЛР №1: парсинг, валидация, коды ошибок, недоступный сервер.

---
## 11. Карта «требование → код → результат»

| Требование | Реализация | Наблюдаемый результат |
|---|---|---|
| Асинхронный приём, Task на клиента | `await _listener.AcceptTcpClientAsync` + `Task.Run(HandleClientAsync)` | сервер принимает клиентов, не дожидаясь обработки |
| Идентификаторы клиентов | `Interlocked.Increment(ref _clientCounter)` | `Client 1`, `Client 2`, ... в логах |
| Журнал START/END с метками | `Log(...)` -> `Journal.Enqueue` + `Console.WriteLine` | строки вида `[10:15:01.120] Client 1 -> START CreateOrder Task=4 Thread=6` |
| Экспериментальная задержка | `delayMs` -> `await Task.Delay(delayMs, ct)` в `HandleAsync` | управляемая длительность обработки |
| Асинхронный ввод-вывод | `ReadMessageAsync` / `WriteMessageAsync` | поток не блокируется на сети |
| Доказательство параллельности | `Journal` (перекрытие интервалов) + общее время | ~10 с вместо ~20 с |
| Демонстрация гонки | `ReserveBouquet` (READ -> DELAY -> WRITE) | оба SUCCESS, `AvailableCount = -1` |
| Устойчивость к ошибкам клиента | `catch` в `HandleClientAsync` и в `HandleAsync` | сервер продолжает работать |
| Корректные ID задач и потоков | `Task.CurrentId`, `Environment.CurrentManagedThreadId` | `Task=4 Thread=6` |
| Тесты | `ParallelProcessingTests` + тесты ЛР №1 | `dotnet test` — зелёные |

### 11.1 Термины, которые нужно уметь объяснить

| Термин | Определение в контексте этой ЛР |
|---|---|
| Процесс | отдельная программа (`Server` / `Client`), своя память |
| Поток | `Environment.CurrentManagedThreadId` в журнале |
| Task | `Task.CurrentId`; `Task.Run(HandleClientAsync)` — задача на клиента |
| async/await | синтаксический способ не блокировать поток на I/O и `Task.Delay` |
| Параллельность | перекрытие интервалов `[START, END]` двух клиентов |
| Конкурентность | множество одновременно *существующих* задач на ограниченном числе потоков |
| Потокобезопасность | корректность структуры данных при конкурентном доступе (`ConcurrentQueue`) |
| Race condition | результат зависит от порядка выполнения конкурентных операций |
| Окно гонки | интервал между READ (проверкой) и WRITE (изменением) |
| Потерянное обновление | запись по устаревшему значению; итог `1 -> 0 -> -1` |
| `volatile` | гарантия видимости изменения между потоками |
| `Interlocked` | атомарные операции над примитивами без блокировки |
| `CancellationToken` | кооперативная отмена: `Task.Delay`, `ReadAsync`, `AcceptTcpClientAsync` |

### 11.2 Ключевые выводы

1. **Параллельность в .NET достигается без выделения потока на задачу.**
   Поток — ресурс дорогой и ограниченный; Task — лёгкая единица работы.
   Освобождение потока на `await` и есть механизм масштабирования.
2. **Доказательство параллельности требует измерения, а не утверждения.**
   В этой ЛР измерение проведено двумя независимыми способами: перекрытием
   интервалов в `Journal` и суммарным временем выполнения — в ручном
   эксперименте и в автотестах.
3. **Потокобезопасные коллекции не решают проблему гонки.**
   `ConcurrentQueue` защищает *структуру данных*, но бизнес-операция
   «проверить и изменить» требует атомарности *последовательности*.
4. **Воспроизводимость дефекта — тоже результат.**
   Тест, фиксирующий `availableCount == -1` как ожидаемый результат,
   документирует найденную проблему и предотвращает случайную «починку».
   Реальные механизмы синхронизации — предмет ЛР №3.

---

*Конспект описывает только код решения `Лабы\\Лаб2`. Все фрагменты приведены
дословно из файлов проекта; пояснения отделены от кода и помечены как таковые.*